In [5]:
"""
Paderborn Bearing Dataset — Production Feature Extraction Pipeline
==================================================================
Reads HH/IR/OR CSV files (vibration + 2 phase currents), trims each to
256 000 samples, slices into 125 windows of W = 4 096 samples with
H = 2 048 hop (50 % overlap), and computes 55 physically meaningful
features per window.  The resulting matrix is written to
/mnt/agents/output/paderborn_features.csv.

Notes
-----
* Window-size validation runs first (Rayleigh / impact-count tests).
* 125 windows/file requires 1 zero-padded window (the strict math
  gives 124).  Bias is documented in the validation report.
* Envelope FFT (2× zero-pad) has 7.8125-Hz bin width — wider than
  the ±1.0 Hz tolerance, so a nearest-bin fallback is used.

No ML models, no train/test splits, no feature selection, no
normalisation.  Pure feature extraction.
"""

from __future__ import annotations
import warnings
from pathlib import Path
from typing import Tuple, Dict

import numpy as np
import pandas as pd
from scipy import signal, stats

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Constants (hardcoded per prompt)
# ---------------------------------------------------------------------------
FS: float = 64_000.0
N_TRIM: int = 256_000
W: int = 4_096
H: int = 2_048
N_WIN_FULL: int = (N_TRIM - W) // H + 1   # = 124 (strict math)
N_WIN: int = 125                          # per prompt; last window zero-padded

# Physical fault frequencies (hardcoded per prompt)
FE:   float = 100.0
FR:   float = 24.41
BPFO: float = 3.0530 * FR    #  74.5741 Hz
BPFI: float = 4.9473 * FR    # 120.7790 Hz
FTF:  float = 0.3818 * FR    #   9.3177 Hz
BSF:  float = 1.9918 * FR    #  48.6418 Hz

# I/O paths
INPUT_DIR = Path(
    "/home/shawky/Documents/nti/actual nti/paderborn/"
    "Processed subset used in our STR-DDPM experiments/"
    "paderborn_subset_used_in_this_study/"
    "University of Paderborn Electric Motor Dataset/"
)
FILES = {"HH": "HH.csv", "IR": "IR.csv", "OR": "OR.csv"}
LABEL_MAP = {"HH": 0, "IR": 1, "OR": 2}
# Output path — tries /mnt/agents/output first, then falls back to a
# writable directory near the script / in HOME / in /tmp.
_OUT_CANDIDATES = [
    Path("/mnt/agents/output/paderborn_features.csv"),
    Path.cwd() / "paderborn_features.csv",
    Path.home() / "paderborn_features.csv",
    Path("/tmp/paderborn_features.csv"),
]

def _resolve_out_path() -> Path:
    """Return the first candidate path whose parent is writable."""
    for p in _OUT_CANDIDATES:
        try:
            p.parent.mkdir(parents=True, exist_ok=True)
            # Probe writability with a tiny touch test
            (p.parent / ".write_probe").touch()
            (p.parent / ".write_probe").unlink()
            return p
        except (PermissionError, OSError):
            continue
    # Last resort: cwd
    return Path.cwd() / "paderborn_features.csv"

OUT_PATH: Path = _resolve_out_path()

# Tolerances (Hz)
ENV_TOL    = 1.0    # envelope fault-freq bands ±1 Hz (with nearest-bin fallback)
CUR_TOL_H  = 2.0    # current harmonic bands ±2 Hz
CUR_TOL_SB = 1.5    # current sideband bands ±1.5 Hz

# Pre-computed Butterworth SOS for envelope bandpass (4th-order, 2-12 kHz)
_SOS_ENV = signal.butter(4, [2_000, 12_000], btype="bandpass",
                         fs=FS, output="sos")


# ===========================================================================
# Window-size validation
# ===========================================================================
def validate_window_size() -> bool:
    """Physics-based check that W = 4 096 satisfies all resolution criteria."""
    T = W / FS
    rayleigh = 1.0 / T                # 15.625 Hz
    n_revs = FR * T                   # shaft revolutions per window
    n_bpfo = BPFO * T                 # outer-race impacts per window
    n_bpfi = BPFI * T                 # inner-race impacts per window
    min_sep = FR                      # BPFI vs BPFI±FR spacing

    ok = (rayleigh < min_sep
          and n_revs >= 1.0
          and n_bpfo >= 1.0
          and n_bpfi >= 1.0)

    print("=" * 72)
    print(" WINDOW-SIZE VALIDATION  (W = 4096 samples = 64 ms @ 64 kHz)")
    print("=" * 72)
    print(f"  Window duration          T   = {T*1000:8.3f} ms")
    print(f"  Rayleigh resolution     Δf  = {rayleigh:8.3f} Hz")
    print(f"  Smallest envelope spacing   = {min_sep:8.3f} Hz (FR)")
    print(f"  Shaft revs / window         = {n_revs:8.3f}  (≥1 required)")
    print(f"  BPFO impacts / window       = {n_bpfo:8.3f}  (≥1 required)")
    print(f"  BPFI impacts / window       = {n_bpfi:8.3f}  (≥1 required)")
    print(f"  Welch PSD bin width         = {FS/W:8.3f} Hz (4096-pt)")
    print(f"  Envelope FFT bin width (2×) = {FS/(2*W):8.3f} Hz (8192-pt)")
    print(f"  Resolves BPFI vs BPFI±FR ?  = "
          f"{'YES' if rayleigh < min_sep else 'NO'}")
    print(f"  Verdict                     = "
          f"{'PASS — W = 4096 is adequate' if ok else 'FAIL'}")
    print("=" * 72 + "\n")
    return ok


# ===========================================================================
# Helpers & input validation
# ===========================================================================
def _check_window(x: np.ndarray, expected_len: int = W) -> None:
    """Validate that x is a 1-D numpy array of length `expected_len`."""
    if not isinstance(x, np.ndarray):
        raise TypeError(f"Expected numpy.ndarray, got {type(x).__name__}")
    if x.ndim != 1:
        raise ValueError(f"Expected 1-D array; got ndim={x.ndim}")
    if x.shape[0] != expected_len:
        raise ValueError(f"Expected length {expected_len}; got {x.shape[0]}")
    if not np.all(np.isfinite(x)):
        raise ValueError("Input contains NaN or Inf")


def _trapz(y: np.ndarray, x: np.ndarray) -> float:
    """Trapezoidal integration (NumPy 1.x & 2.x compatible)."""
    if hasattr(np, "trapezoid"):
        return float(np.trapezoid(y, x))
    return float(np.trapz(y, x))


def _integrate_psd(f: np.ndarray, P: np.ndarray,
                   lo: float, hi: float) -> float:
    """Integrate one-sided PSD over [lo, hi] Hz. 0.0 if empty."""
    m = (f >= lo) & (f <= hi)
    if not np.any(m):
        return 0.0
    return _trapz(P[m], f[m])


def _band_max(f: np.ndarray, P: np.ndarray,
              center: float, tol: float) -> float:
    """Max(PSD) in [center-tol, center+tol]. 0.0 if empty."""
    m = (f >= center - tol) & (f <= center + tol)
    if not np.any(m):
        return 0.0
    return float(P[m].max())


def _sum_magnitude(f_env: np.ndarray, mag: np.ndarray,
                   center: float, tol: float) -> float:
    """Sum |FFT(envelope)| bins within ±tol Hz of `center`.

    If no bin centre falls inside ±tol (which is possible because the
    8192-pt FFT has bin width 7.8125 Hz, wider than 2·1.0 Hz), fall
    back to the nearest bin's magnitude.  This guarantees a physically
    meaningful, non-NaN value for every fault-frequency feature.
    """
    m = (f_env >= center - tol) & (f_env <= center + tol)
    if np.any(m):
        return float(mag[m].sum())
    idx = int(np.argmin(np.abs(f_env - center)))
    return float(mag[idx])


def _welch_psd(x: np.ndarray,
               nperseg: int = W) -> Tuple[np.ndarray, np.ndarray]:
    """One-sided Welch PSD (Hann, no overlap, nfft = nperseg = 4096)."""
    _check_window(x)
    f, P = signal.welch(
        x - x.mean(),
        fs=FS,
        window="hann",
        nperseg=nperseg,
        nfft=nperseg,
        noverlap=0,
        return_onesided=True,
        scaling="density",
    )
    return f, P


# ===========================================================================
# 1. Vibration — Time Domain (11 features)
# ===========================================================================
def vib_time_features(x: np.ndarray) -> Dict[str, float]:
    """11 time-domain vibration features on a zero-mean window.

    Fisher (excess) kurtosis; Fisher–Pearson moment skewness.
    """
    _check_window(x)
    x = x - x.mean()
    abs_x = np.abs(x)
    sqrt_abs = np.sqrt(abs_x)

    rms         = float(np.sqrt(np.mean(x ** 2)))
    mean_abs    = float(abs_x.mean())
    mean_sqrt   = float(sqrt_abs.mean())
    peak        = float(abs_x.max())
    p2p         = float(x.max() - x.min())
    std         = float(x.std())
    eps         = 1e-12

    return {
        "vib_mean":      float(x.mean()),
        "vib_std":       std,
        "vib_rms":       rms,
        "vib_peak":      peak,
        "vib_p2p":       p2p,
        "vib_skew":      float(stats.skew(x, bias=True)),
        "vib_kurt":      float(stats.kurtosis(x, fisher=True, bias=True)),
        "vib_crest":     peak / (rms + eps),
        "vib_clearance": peak / ((mean_sqrt) ** 2 + eps),
        "vib_shape":     rms / (mean_abs + eps),
        "vib_impulse":   peak / (mean_abs + eps),
    }


# ===========================================================================
# 2. Vibration — Frequency Domain (10 features, Welch PSD)
# ===========================================================================
def vib_freq_features(x: np.ndarray) -> Dict[str, float]:
    """10 frequency-domain vibration features."""
    f, P = _welch_psd(x)

    p0_1k   = _integrate_psd(f, P, 0,       1_000)
    p1_5k   = _integrate_psd(f, P, 1_000,   5_000)
    p5_10k  = _integrate_psd(f, P, 5_000,  10_000)
    p10_20k = _integrate_psd(f, P, 10_000, 20_000)
    p20_32k = _integrate_psd(f, P, 20_000, 32_000)

    total   = _trapz(P, f)
    centroid = _trapz(f * P, f) / (total + 1e-15)

    p_norm = P / (P.sum() + 1e-15)
    p_norm = np.where(p_norm > 0, p_norm, 1e-15)
    entropy = float(-np.sum(p_norm * np.log(p_norm)))

    cum = np.cumsum(P)
    if cum[-1] <= 0:
        rolloff = 0.0
    else:
        cutoff = 0.85 * cum[-1]
        idx = int(np.searchsorted(cum, cutoff))
        rolloff = float(f[min(idx, len(f) - 1)])

    m = f <= 10_000
    if np.any(m) and P[m].max() > 0:
        dom_freq = float(f[m][np.argmax(P[m])])
    else:
        dom_freq = 0.0

    return {
        "vib_power_0_1k":            p0_1k,
        "vib_power_1_5k":            p1_5k,
        "vib_power_5_10k":           p5_10k,
        "vib_power_10_20k":          p10_20k,
        "vib_power_20_32k":          p20_32k,
        "vib_spectral_centroid":     centroid,
        "vib_spectral_entropy":      entropy,
        "vib_spectral_rolloff_85":   rolloff,
        "vib_dominant_freq":         dom_freq,
        "vib_band_ratio_5_10_0_1":   p5_10k / (p0_1k + 1e-12),
    }


# ===========================================================================
# 3. Vibration — Envelope Demodulation (13 features)
# ===========================================================================
def vib_envelope_features(x: np.ndarray) -> Dict[str, float]:
    """13 envelope-spectrum features.

    Pipeline (per prompt):
      bandpass(2-12 kHz, Butterworth 4, zero-phase)  →  |Hilbert|  →
      subtract mean  →  rFFT zero-padded to 2*W = 8192  →  |X|.
    """
    _check_window(x)
    x = x - x.mean()
    filtered = signal.sosfiltfilt(_SOS_ENV, x)
    envelope = np.abs(signal.hilbert(filtered))
    env_mc   = envelope - envelope.mean()

    n_fft  = 2 * W
    mag    = np.abs(np.fft.rfft(env_mc, n=n_fft))
    f_env  = np.fft.rfftfreq(n_fft, d=1.0 / FS)

    return {
        "env_bpfo_1x":       _sum_magnitude(f_env, mag, 1 * BPFO, ENV_TOL),
        "env_bpfo_2x":       _sum_magnitude(f_env, mag, 2 * BPFO, ENV_TOL),
        "env_bpfo_3x":       _sum_magnitude(f_env, mag, 3 * BPFO, ENV_TOL),
        "env_bpfi_1x":       _sum_magnitude(f_env, mag, 1 * BPFI, ENV_TOL),
        "env_bpfi_2x":       _sum_magnitude(f_env, mag, 2 * BPFI, ENV_TOL),
        "env_bpfi_3x":       _sum_magnitude(f_env, mag, 3 * BPFI, ENV_TOL),
        "env_bpfi_sb_plus":  _sum_magnitude(f_env, mag, BPFI + FR, ENV_TOL),
        "env_bpfi_sb_minus": _sum_magnitude(f_env, mag, BPFI - FR, ENV_TOL),
        "env_bpfo_sb_plus":  _sum_magnitude(f_env, mag, BPFO + FR, ENV_TOL),
        "env_bpfo_sb_minus": _sum_magnitude(f_env, mag, BPFO - FR, ENV_TOL),
        "env_rms":   float(np.sqrt(np.mean(env_mc ** 2))),
        "env_kurt":  float(stats.kurtosis(env_mc, fisher=True)),
        "env_peak":  float(np.max(np.abs(env_mc))),
    }


# ===========================================================================
# 4. Current — Time & Frequency (10 features per channel)
# ===========================================================================
def current_features(x: np.ndarray, prefix: str) -> Dict[str, float]:
    """10 features per current channel (i1 / i2)."""
    _check_window(x)
    x = x - x.mean()
    rms  = float(np.sqrt(np.mean(x ** 2)))
    std  = float(x.std())
    peak = float(np.max(np.abs(x)))
    f, P = _welch_psd(x)

    p50  = _integrate_psd(f, P, 50  - CUR_TOL_H, 50  + CUR_TOL_H)
    p100 = _integrate_psd(f, P, 100 - CUR_TOL_H, 100 + CUR_TOL_H)
    p150 = _integrate_psd(f, P, 150 - CUR_TOL_H, 150 + CUR_TOL_H)

    def _sb(c: float) -> float:
        """Peak PSD at FE + c and FE - c, summed."""
        return (_band_max(f, P, FE + c, CUR_TOL_SB) +
                _band_max(f, P, FE - c, CUR_TOL_SB))

    return {
        f"{prefix}_rms":              rms,
        f"{prefix}_std":              std,
        f"{prefix}_peak":             peak,
        f"{prefix}_power_50hz":       p50,
        f"{prefix}_power_100hz":      p100,
        f"{prefix}_power_150hz":      p150,
        f"{prefix}_sideband_fr_1x":   _sb(1 * FR),
        f"{prefix}_sideband_fr_2x":   _sb(2 * FR),
        f"{prefix}_sideband_bpfo":    _sb(BPFO),
        f"{prefix}_sideband_bpfi":    _sb(BPFI),
    }


# ===========================================================================
# 5. Cross-sensor coherence (1 feature)
# ===========================================================================
def vib_i1_coherence(vib: np.ndarray, i1: np.ndarray) -> float:
    """Mean |C(vib, i1)|² over 0–2000 Hz (Welch nperseg = 2048, Hann)."""
    _check_window(vib)
    _check_window(i1)
    f, C = signal.coherence(
        vib - vib.mean(),
        i1 - i1.mean(),
        fs=FS,
        window="hann",
        nperseg=2_048,
        noverlap=1_024,
    )
    m = (f >= 0) & (f <= 2_000)
    if not np.any(m):
        return 0.0
    return float(C[m].mean())


# ===========================================================================
# Driver: load, slice, extract, save
# ===========================================================================
def load_signal(name: str) -> Dict[str, np.ndarray]:
    """Load CSV, trim to N_TRIM samples, return per-channel float64 arrays."""
    path = INPUT_DIR / FILES[name]
    if not path.exists():
        raise FileNotFoundError(f"Cannot find {path}")
    df = pd.read_csv(path)
    return {ch: df[ch].to_numpy(dtype=np.float64)[:N_TRIM]
            for ch in df.columns}


def _window_slice(arr: np.ndarray, w_idx: int) -> np.ndarray:
    """Return window `w_idx` of `arr`, zero-padding the tail if needed."""
    start = w_idx * H
    end   = start + W
    if end <= arr.shape[0]:
        return arr[start:end].copy()
    out = np.zeros(W, dtype=arr.dtype)
    n_real = arr.shape[0] - start
    out[:n_real] = arr[start:]
    return out


def extract_window(vib: np.ndarray, i1: np.ndarray,
                   i2: np.ndarray) -> Dict[str, float]:
    """Compute all 55 features for a single 4096-sample window."""
    feats: Dict[str, float] = {}
    feats.update(vib_time_features(vib))
    feats.update(vib_freq_features(vib))
    feats.update(vib_envelope_features(vib))
    feats.update(current_features(i1, "i1"))
    feats.update(current_features(i2, "i2"))
    feats["vib_i1_coherence_mean_0_2k"] = vib_i1_coherence(vib, i1)
    return feats


def build_features_df() -> pd.DataFrame:
    """3 files × 125 windows × 55 features + 3 metadata columns."""
    rows = []
    for name in ("HH", "IR", "OR"):
        sig = load_signal(name)
        for w_idx in range(N_WIN):
            row: Dict[str, object] = {
                "label":       LABEL_MAP[name],
                "source_file": name,
                "window_id":   w_idx,
            }
            row.update(extract_window(
                _window_slice(sig["vibration"],       w_idx),
                _window_slice(sig["phase_current_1"], w_idx),
                _window_slice(sig["phase_current_2"], w_idx),
            ))
            rows.append(row)
        n_full = min(N_WIN, N_WIN_FULL)
        n_pad  = max(0, N_WIN - N_WIN_FULL)
        print(f"  [{name}] {n_full} full windows + "
              f"{n_pad} zero-padded = {N_WIN} total")
    return pd.DataFrame(rows)


# ===========================================================================
# Validation report
# ===========================================================================
def validation_report(df: pd.DataFrame) -> None:
    print("\n" + "=" * 72)
    print(" VALIDATION REPORT")
    print("=" * 72)
    print(f"  Total rows             : {len(df)}")
    print(f"  Total columns          : {df.shape[1]}")
    expected_rows = 3 * N_WIN
    ok_shape = (len(df) == expected_rows and df.shape[1] == 58)
    print(f"  Expected ({expected_rows} × 58)   : "
          f"{'PASS' if ok_shape else 'FAIL'}")

    feat_cols = [c for c in df.columns
                 if c not in ("label", "source_file", "window_id")]
    feat_arr = df[feat_cols].to_numpy(dtype=np.float64)
    n_nan = int(np.isnan(feat_arr).sum())
    n_inf = int(np.isinf(feat_arr).sum())
    print(f"  Total NaN (features)   : {n_nan}")
    print(f"  Total Inf (features)   : {n_inf}")
    if n_nan > 0:
        nan_cols = [c for c in feat_cols if df[c].isna().any()]
        print(f"  NaN columns            : {nan_cols}")
    if n_inf > 0:
        inf_cols = [c for c in feat_cols
                    if np.isinf(df[c].to_numpy(dtype=np.float64)).any()]
        print(f"  Inf columns            : {inf_cols}")

    print("\n  Windows per class:")
    for name, lab in LABEL_MAP.items():
        cnt = int((df["label"] == lab).sum())
        print(f"    {name} (label={lab}): {cnt}")

    print("\n  vib_rms per class:")
    for name, lab in LABEL_MAP.items():
        sub = df.loc[df["label"] == lab, "vib_rms"]
        print(f"    {name} (label={lab}):  "
              f"mean={sub.mean():.5f}   std={sub.std():.5f}")

    hh_bpfi = df.loc[df["label"] == 0, "env_bpfi_1x"].mean()
    ir_bpfi = df.loc[df["label"] == 1, "env_bpfi_1x"].mean()
    hh_bpfo = df.loc[df["label"] == 0, "env_bpfo_1x"].mean()
    or_bpfo = df.loc[df["label"] == 2, "env_bpfo_1x"].mean()
    print(f"\n  env_bpfi_1x mean — HH: {hh_bpfi:.5f} | IR: {ir_bpfi:.5f}")
    print(f"    IR > HH ? {'YES ✓' if ir_bpfi > hh_bpfi else 'NO ✗'}")
    print(f"  env_bpfo_1x mean — HH: {hh_bpfo:.5f} | OR: {or_bpfo:.5f}")
    print(f"    OR > HH ? {'YES ✓' if or_bpfo > hh_bpfo else 'NO ✗'}")
    print("=" * 72)


# ===========================================================================
# Entry point
# ===========================================================================
def main() -> None:
    validate_window_size()
    df = build_features_df()
    OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUT_PATH, index=False)
    print(f"\n  Saved → {OUT_PATH}  "
          f"({df.shape[0]} × {df.shape[1]})")
    validation_report(df)


if __name__ == "__main__":
    main()

 WINDOW-SIZE VALIDATION  (W = 4096 samples = 64 ms @ 64 kHz)
  Window duration          T   =   64.000 ms
  Rayleigh resolution     Δf  =   15.625 Hz
  Smallest envelope spacing   =   24.410 Hz (FR)
  Shaft revs / window         =    1.562  (≥1 required)
  BPFO impacts / window       =    4.770  (≥1 required)
  BPFI impacts / window       =    7.729  (≥1 required)
  Welch PSD bin width         =   15.625 Hz (4096-pt)
  Envelope FFT bin width (2×) =    7.812 Hz (8192-pt)
  Resolves BPFI vs BPFI±FR ?  = YES
  Verdict                     = PASS — W = 4096 is adequate

  [HH] 124 full windows + 1 zero-padded = 125 total
  [IR] 124 full windows + 1 zero-padded = 125 total
  [OR] 124 full windows + 1 zero-padded = 125 total

  Saved → /home/shawky/Documents/nti/actual nti/paderborn_features.csv  (375 × 58)

 VALIDATION REPORT
  Total rows             : 375
  Total columns          : 58
  Expected (375 × 58)   : PASS
  Total NaN (features)   : 0
  Total Inf (features)   : 0

  Windows per cla